In [1]:
import pandas as pd
import numpy as np
import datetime as dt



In [2]:
def clean_transaction_data(df):
    """Clean the raw transaction data"""
    # Convert date columns to datetime
    df['InvoiceDate'] = pd.to_datetime(df['InvoiceDate'])

    # Extract date components
    df['InvoiceYear'] = df['InvoiceDate'].dt.year
    df['InvoiceMonth'] = df['InvoiceDate'].dt.month
    df['InvoiceDay'] = df['InvoiceDate'].dt.day
    df['InvoiceDayOfWeek'] = df['InvoiceDate'].dt.dayofweek
    df['InvoiceHour'] = df['InvoiceDate'].dt.hour

    # Create TotalAmount column
    df['TotalAmount'] = df['Quantity'] * df['Price']

    # Filter out records with missing CustomerID
    df = df.dropna(subset=['CustomerID'])

    # Convert CustomerID to string type
    df['CustomerID'] = df['CustomerID'].astype(str)

    # Filter out cancelled orders (negative quantities)
    df = df[df['Quantity'] > 0]

    # Filter out records with zero or negative price
    df = df[df['Price'] > 0]

    # Handle outliers
    quantity_threshold = df['Quantity'].quantile(0.99)
    price_threshold = df['Price'].quantile(0.99)
    df = df[(df['Quantity'] <= quantity_threshold) & (df['Price'] <= price_threshold)]

    return df

In [3]:
def create_rfm_features(df, reference_date=None):
    """Create RFM (Recency, Frequency, Monetary) features"""
    if reference_date is None:
        reference_date = df['InvoiceDate'].max() + dt.timedelta(days=1)

    # Group by customer
    rfm = df.groupby('CustomerID').agg({
        'InvoiceDate': lambda x: (reference_date - x.max()).days,  # Recency
        'Invoice': 'nunique',  # Frequency
        'TotalAmount': 'sum'  # Monetary
    })

    # Rename columns
    rfm.columns = ['Recency', 'Frequency', 'Monetary']

    return rfm

In [4]:
def create_customer_features(df, reference_date=None):
    """Create comprehensive customer features for modeling"""
    if reference_date is None:
        reference_date = df['InvoiceDate'].max()

    # Aggregate customer data
    customer_features = df.groupby('CustomerID').agg({
        'Invoice': 'nunique',  # Number of orders
        'TotalAmount': ['sum', 'mean'],  # Total and average purchase amount
        'Quantity': ['sum', 'mean'],  # Total and average quantity ordered
        'InvoiceDate': [
            lambda x: (reference_date - x.max()).days,  # Recency
            lambda x: (x.max() - x.min()).days if len(x) > 1 else 0,  # Customer age (days between first and last purchase)
            'count'  # Number of items purchased
        ]
    })

    # Flatten column names
    customer_features.columns = ['NumOrders', 'TotalAmount', 'AvgOrderValue',
                               'TotalQuantity', 'AvgQuantity', 'Recency', 'CustomerAge', 'NumItems']

    # Reset index to make CustomerID a column
    customer_features = customer_features.reset_index()

    # Calculate purchases frequency (orders per week)
    customer_features['PurchaseFrequency'] = customer_features['NumOrders'] / ((customer_features['CustomerAge'] / 7) + 0.001)

    # Calculate purchase interval (average days between orders)
    customer_features['AvgPurchaseInterval'] = customer_features['CustomerAge'] / (customer_features['NumOrders'] + 0.001)

    return customer_features


In [5]:
def define_churn(df, threshold_days=90):
    """Define churn based on recency"""
    max_date = df['InvoiceDate'].max()
    churn_date = max_date - dt.timedelta(days=threshold_days)

    # Get last purchase date for each customer
    last_purchase_date = df.groupby('CustomerID')['InvoiceDate'].max().reset_index()
    last_purchase_date.columns = ['CustomerID', 'LastPurchaseDate']

    # Define churn (1 if customer hasn't purchased in the last threshold_days)
    last_purchase_date['Churned'] = (last_purchase_date['LastPurchaseDate'] < churn_date).astype(int)

    return last_purchase_date